# Introduction to Neural Networks

**Junior Design**

---

## The Big Picture

Every ML algorithm you've learned so far — linear regression, logistic regression, decision trees, random forests, SVMs — was designed by humans based on mathematical intuition about how to separate classes or fit curves.

**Neural networks take a different approach.** Instead of designing the right mathematical function, we design a flexible *architecture* that can learn *any* function from data. We give it building blocks (called **neurons**), connect them together, and let gradient descent figure out the details.

This flexibility is both a strength and a weakness of neural networks. They can learn patterns that no hand-designed algorithm would find — but they need more data, more computation, and more careful tuning.

###  Breakdown

| Topic | Description |
|---|---|
| **Part 1** | What is a neuron? (Spoiler: you already know) |
| **Part 2** | Activation functions: why nonlinearity matters |
| **Part 3** | Building a network: layers, weights, and architecture |
| **Part 4** | How networks learn: forward pass & backpropagation |
| **Part 5** | Watching a network train |
| **Part 6** | Practical considerations: scaling, overfitting, architecture |
| **Part 7** | Real application: handwritten digit recognition |
| **Part 8** | Real application: breast cancer classification |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.colors import ListedColormap

from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score
)
from sklearn.datasets import (
    make_moons, make_circles, make_classification,
    load_breast_cancer, load_digits
)

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("All imports successful!")

In [ ]:
# ============================================================
# Helper: plot 2D decision boundaries (used throughout)
# ============================================================
def plot_decision_boundary(model, X, y, ax=None, title='', h=0.02):
    """Plot the decision boundary of a 2D classifier."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
    cmap_bold = ListedColormap(['#FF0000', '#0000FF'])
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold, 
              edgecolors='black', s=40, alpha=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    return ax

print("Helper function ready.")

---

## Part 1: What Is a Neuron?

### The Biological Inspiration (Very Loosely)

A biological neuron receives electrical signals through its **dendrites**, processes them in the **cell body**, and if the combined signal is strong enough, it fires an output through its **axon** to other neurons.

An artificial neuron does something analogous (but much simpler):

1. **Receives inputs** ($x_1, x_2, \ldots, x_n$) — these are your features
2. **Multiplies each input by a weight** ($w_1 x_1 + w_2 x_2 + \ldots + w_n x_n$) — some inputs matter more than others
3. **Adds a bias** ($+ b$) — a baseline offset
4. **Applies an activation function** $f(z)$ — decides whether and how strongly to "fire"
5. **Produces an output** — which becomes input to the next layer

$$\text{output} = f\left(\sum_{i=1}^{n} w_i x_i + b\right) = f(\vec{w} \cdot \vec{x} + b)$$

### Here's the Secret: You Already Know This

Look at logistic regression:

$$\hat{y} = \sigma(w_1 x_1 + w_2 x_2 + \ldots + w_n x_n + b)$$

That's exactly a single neuron with the sigmoid activation function. **Logistic regression IS a one-neuron neural network.**

In [ ]:
# ============================================================
# Visualize: what a single neuron computes
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Left: Diagram of a single neuron ---
ax = axes[0]
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-1.5, 3.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Anatomy of a Single Neuron', fontweight='bold', fontsize=13)

# Input nodes
input_labels = ['$x_1$', '$x_2$', '$x_3$']
input_y = [2.5, 1.0, -0.5]
for i, (label, iy) in enumerate(zip(input_labels, input_y)):
    circle = plt.Circle((0.5, iy), 0.3, color='lightblue', ec='black', linewidth=2)
    ax.add_patch(circle)
    ax.text(0.5, iy, label, ha='center', va='center', fontsize=14)

# Neuron body
neuron = plt.Circle((2.5, 1.0), 0.5, color='lightyellow', ec='black', linewidth=2)
ax.add_patch(neuron)
ax.text(2.5, 1.0, '$\Sigma \\rightarrow f$', ha='center', va='center', fontsize=14)

# Arrows with weights
weight_labels = ['$w_1$', '$w_2$', '$w_3$']
for iy, wl in zip(input_y, weight_labels):
    ax.annotate('', xy=(2.0, 1.0), xytext=(0.8, iy),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='steelblue'))
    mid_x = 1.4
    mid_y = (iy + 1.0) / 2 + 0.15
    ax.text(mid_x, mid_y, wl, fontsize=11, color='steelblue', fontweight='bold')

# Bias
ax.annotate('', xy=(2.5, 0.5), xytext=(2.5, -0.5),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='green'))
ax.text(2.7, -0.3, '$b$ (bias)', fontsize=11, color='green', fontweight='bold')

# Output arrow
ax.annotate('', xy=(4.0, 1.0), xytext=(3.0, 1.0),
            arrowprops=dict(arrowstyle='->', lw=2, color='black'))
ax.text(4.1, 1.0, 'output', fontsize=12, va='center')

# Formula
ax.text(2.5, 3.2, '$\mathrm{output} = f(w_1 x_1 + w_2 x_2 + w_3 x_3 + b)$', 
        ha='center', fontsize=13, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# --- Right: What it looks like in practice ---
ax2 = axes[1]
np.random.seed(42)

# A neuron with 2 inputs (so we can plot it)
X_demo = np.random.randn(100, 2)
# Weights and bias (pretend these are learned)
w = np.array([1.5, -1.0])
b = 0.3
z = X_demo @ w + b
sigmoid = 1 / (1 + np.exp(-z))
y_demo = (sigmoid > 0.5).astype(int)

# Plot the classification
cmap = ListedColormap(['#FF6B6B', '#4ECDC4'])
scatter = ax2.scatter(X_demo[:, 0], X_demo[:, 1], c=sigmoid, cmap='RdYlGn', 
                       edgecolors='black', s=60, alpha=0.8)
plt.colorbar(scatter, ax=ax2, label='Neuron output (0 to 1)')

# Draw the decision boundary: w1*x1 + w2*x2 + b = 0 → x2 = -(w1*x1 + b)/w2
x_line = np.linspace(-3, 3, 100)
y_line = -(w[0] * x_line + b) / w[1]
ax2.plot(x_line, y_line, 'k--', linewidth=2, label='Decision boundary')
ax2.set_xlim(-3, 3)
ax2.set_ylim(-3, 3)
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.set_title('What a Single Neuron Computes\n(= a straight line boundary)', fontweight='bold', fontsize=13)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("A single neuron can only draw a straight line (or hyperplane) through feature space.")
print("This is exactly what logistic regression does!")
print("To handle complex, curved boundaries, we need MULTIPLE neurons working together.")

In [ ]:
# ============================================================
# Prove it: single neuron ≈ logistic regression
# ============================================================
X_simple, y_simple = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=2.0, random_state=42
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logistic Regression
lr = LogisticRegression()
lr.fit(X_simple, y_simple)
plot_decision_boundary(lr, X_simple, y_simple, ax=axes[0],
                       title=f'Logistic Regression\nAccuracy: {lr.score(X_simple, y_simple):.1%}')

# Neural network with a tiny hidden layer + sigmoid
# Note: we use 3 neurons rather than 1, because MLPClassifier always adds
# a separate output layer on top. With only 1 hidden neuron, the network
# squeezes 2D input through a 1D bottleneck (sigmoid outputs a single
# scalar), and the output layer can't recover the lost information.
# 3 neurons is still tiny, but avoids that degenerate case.
nn_tiny = MLPClassifier(hidden_layer_sizes=(3,), activation='logistic',
                         max_iter=2000, random_state=42)
nn_tiny.fit(X_simple, y_simple)
plot_decision_boundary(nn_tiny, X_simple, y_simple, ax=axes[1],
                       title=f'Tiny Neural Net (3 neurons, sigmoid)\nAccuracy: {nn_tiny.score(X_simple, y_simple):.1%}')

plt.suptitle('A Single Neuron ≈ Logistic Regression', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Nearly identical boundaries! The tiny neural net and logistic regression")
print("are solving the same problem in essentially the same way.")
print("\nSo why do we need neural networks? Because real data isn't this simple...")

In [ ]:
# ============================================================
# ...and a single neuron fails on nonlinear data
# ============================================================
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)

fig, ax = plt.subplots(figsize=(8, 6))

nn_tiny_moons = MLPClassifier(hidden_layer_sizes=(3,), activation='logistic',
                            max_iter=2000, random_state=42)
nn_tiny_moons.fit(X_moons, y_moons)
plot_decision_boundary(nn_tiny_moons, X_moons, y_moons, ax=ax,
                       title=f'Tiny Net (3 sigmoid neurons) on Nonlinear Data — Accuracy: {nn_tiny_moons.score(X_moons, y_moons):.1%}')
plt.tight_layout()
plt.show()

print("A tiny sigmoid network draws a nearly-linear boundary through curved data — it can't")
print("capture the moon shapes. We need more neurons and ReLU activation.")
print("\nThis motivates using BIGGER hidden layers and BETTER activation functions.")

---

## Part 2: Activation Functions — Why Nonlinearity Matters

Before we add more neurons, we need to understand **activation functions** — the $f(z)$ inside each neuron.

### Why Do We Need Them?

Without an activation function, a neuron just computes $z = \vec{w} \cdot \vec{x} + b$, which is a linear function. If you stack multiple layers of linear functions, you still get a linear function:

$$\text{Layer 2}(\text{Layer 1}(\vec{x})) = W_2(W_1 \vec{x} + b_1) + b_2 = (W_2 W_1)\vec{x} + (W_2 b_1 + b_2) = W'\vec{x} + b'$$

That's just linear regression with extra steps. No matter how many layers you add, the result collapses to a single linear transformation.

**Activation functions break the linearity.** By applying a nonlinear function after each layer's linear computation, the network can learn curves, corners, and complex shapes that no single line could capture.

In [ ]:
# ============================================================
# The four activation functions you'll encounter
# ============================================================
z = np.linspace(-5, 5, 300)

activations = [
    ('Sigmoid', 
     1 / (1 + np.exp(-z)),
     'Squashes output to (0, 1).\nHistorically popular, but has\nvanishing gradient problems.',
     '#e74c3c'),
    ('Tanh', 
     np.tanh(z),
     'Squashes to (-1, 1).\nCentered at 0 (better than sigmoid).\nStill has vanishing gradient.',
     '#3498db'),
    ('ReLU', 
     np.maximum(0, z),
     'Dead simple: max(0, z).\nFast, no vanishing gradient.\nTHE default choice for hidden layers.',
     '#2ecc71'),
    ('Leaky ReLU', 
     np.where(z > 0, z, 0.01 * z),
     'Like ReLU but allows small\nnegative gradients. Fixes the\n"dead neuron" problem.',
     '#9b59b6'),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (name, values, desc, color) in zip(axes, activations):
    ax.plot(z, values, linewidth=3, color=color)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('Input (z)', fontsize=11)
    ax.set_ylabel('Output f(z)', fontsize=11)
    ax.text(0.05, 0.95, desc, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', 
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    ax.set_ylim(-2, 5)
    ax.grid(True, alpha=0.3)

plt.suptitle('Activation Functions — The Source of Nonlinearity', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# What "vanishing gradient" means (and why ReLU fixes it)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sigmoid and its derivative
sigmoid = 1 / (1 + np.exp(-z))
sigmoid_grad = sigmoid * (1 - sigmoid)

axes[0].plot(z, sigmoid, linewidth=2.5, label='sigmoid(z)', color='#e74c3c')
axes[0].plot(z, sigmoid_grad, linewidth=2.5, label="sigmoid'(z) (gradient)", 
             color='#e74c3c', linestyle='--')
axes[0].fill_between(z, 0, sigmoid_grad, alpha=0.1, color='#e74c3c')
axes[0].set_title('Sigmoid: Gradient Vanishes at Extremes', fontweight='bold')
axes[0].set_xlabel('z')
axes[0].legend(fontsize=10)
axes[0].annotate('Max gradient = 0.25!', xy=(0, 0.25), xytext=(2, 0.35),
                  arrowprops=dict(arrowstyle='->', color='black'),
                  fontsize=11, fontweight='bold')

# ReLU and its derivative
relu = np.maximum(0, z)
relu_grad = (z > 0).astype(float)

axes[1].plot(z, relu, linewidth=2.5, label='ReLU(z)', color='#2ecc71')
axes[1].plot(z, relu_grad, linewidth=2.5, label="ReLU'(z) (gradient)", 
             color='#2ecc71', linestyle='--')
axes[1].set_title('ReLU: Gradient is 0 or 1 (no vanishing!)', fontweight='bold')
axes[1].set_xlabel('z')
axes[1].set_ylim(-0.5, 5)
axes[1].legend(fontsize=10)
axes[1].annotate('Gradient = 1 (full signal!)', xy=(3, 1), xytext=(1, 2),
                  arrowprops=dict(arrowstyle='->', color='black'),
                  fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("The vanishing gradient problem:")
print("  Sigmoid's max gradient is only 0.25. In backpropagation, gradients")
print("  multiply through layers: 0.25 × 0.25 × 0.25 = 0.016 after just 3 layers.")
print("  The signal to early layers becomes tiny → they stop learning.")
print("\n  ReLU's gradient is 1 for positive inputs — the signal passes through unchanged.")
print("  This is why ReLU enabled training of much deeper networks.")

### Which Activation Function Should You Use?

| Where | What to Use | Why |
|---|---|---|
| **Hidden layers** | ReLU (default) | Fast, avoids vanishing gradient |
| **Output (binary classification)** | Sigmoid | Gives probability between 0 and 1 |
| **Output (multi-class)** | Softmax | Gives probability distribution over all classes |
| **Output (regression)** | Linear (no activation) | Unrestricted output range |

In sklearn's `MLPClassifier`, this is handled for you — you only choose the hidden layer activation via the `activation` parameter.

---

## Part 3: Building a Network — Layers, Weights, and Architecture

Now we connect multiple neurons together into a **network**.

### Vocabulary

| Term | Meaning |
|---|---|
| **Input layer** | One node per feature. Not really "neurons" — they just pass data through. |
| **Hidden layer** | Layer(s) between input and output where the actual learning happens. Called "hidden" because you don't directly observe their values — only the network sees them. |
| **Output layer** | Produces the final prediction. One neuron for binary classification, one per class for multi-class. |
| **Weights** | The $w$ values on each connection. These are what the network *learns*. |
| **Biases** | The $b$ value in each neuron. Also learned. |
| **Parameters** | Total count of all weights + biases. More parameters = more capacity. |
| **Architecture** | The number and size of hidden layers (e.g., "two hidden layers with 64 and 32 neurons"). |

### How Information Flows

```
Features        Hidden Layer 1       Hidden Layer 2       Output
(n inputs)      (e.g., 10 neurons)   (e.g., 5 neurons)   (1 or k neurons)

  x₁ ─────────── h₁ ─────────────── h₆ ─────────────── ŷ
  x₂ ─────────── h₂ ─────────────── h₇
  x₃ ─────────── h₃ ─────────────── h₈
  x₄ ─────────── h₄ ─────────────── h₉
  x₅ ─────────── h₅ ─────────────── h₁₀

  Every node in one layer connects to EVERY node in the next.
  These are called "fully connected" or "dense" layers.
```

### Counting Parameters

If you have 5 inputs → 10 hidden neurons → 1 output:
- Layer 1: (5 inputs × 10 neurons) + 10 biases = **60 parameters**
- Layer 2: (10 inputs × 1 output) + 1 bias = **11 parameters**
- **Total: 71 parameters** that need to be learned from data

In [ ]:
# ============================================================
# Visualize a network architecture
# ============================================================
def draw_network(layer_sizes, ax, title=''):
    """Draw a simple neural network diagram."""
    n_layers = len(layer_sizes)
    max_neurons = max(layer_sizes)
    
    layer_labels = ['Input'] + [f'Hidden {i+1}' for i in range(n_layers - 2)] + ['Output']
    colors = ['lightblue'] + ['lightyellow'] * (n_layers - 2) + ['lightgreen']
    
    positions = {}  # (layer, neuron) -> (x, y)
    
    for i, (size, label, color) in enumerate(zip(layer_sizes, layer_labels, colors)):
        x = i * 2
        # Center neurons vertically
        y_start = (max_neurons - size) / 2
        for j in range(size):
            y = y_start + j
            positions[(i, j)] = (x, y)
            circle = plt.Circle((x, y), 0.3, color=color, ec='black', linewidth=1.5, zorder=3)
            ax.add_patch(circle)
        
        ax.text(x, -1.2, f'{label}\n({size})', ha='center', fontsize=10, fontweight='bold')
    
    # Draw connections
    for i in range(n_layers - 1):
        for j in range(layer_sizes[i]):
            for k in range(layer_sizes[i + 1]):
                x1, y1 = positions[(i, j)]
                x2, y2 = positions[(i + 1, k)]
                ax.plot([x1 + 0.3, x2 - 0.3], [y1, y2], 
                       color='gray', linewidth=0.5, alpha=0.4, zorder=1)
    
    # Count parameters
    total_params = sum(layer_sizes[i] * layer_sizes[i+1] + layer_sizes[i+1] 
                       for i in range(n_layers - 1))
    
    ax.set_xlim(-1, (n_layers - 1) * 2 + 1)
    ax.set_ylim(-2, max_neurons + 0.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f'{title}\nTotal parameters: {total_params:,}', 
                 fontweight='bold', fontsize=12)

# Show three different architectures
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

draw_network([4, 3, 1], axes[0], 'Small Network')
draw_network([4, 6, 4, 1], axes[1], 'Medium Network')
draw_network([4, 8, 6, 4, 1], axes[2], 'Deep Network')

plt.suptitle('Neural Network Architectures — More Layers and Neurons = More Parameters', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Each line represents a weight that must be learned from data.")
print("Each colored circle is a neuron with its own bias.")
print("\nMore parameters = more capacity to learn complex patterns...")
print("           ...but also more risk of memorizing noise (overfitting).")

In [ ]:
# ============================================================
# Watch what happens as we add neurons: boundaries get complex
# ============================================================
architectures = [
    ((3,),          '3 neurons'),
    ((5,),          '5 neurons'),
    ((10,),         '10 neurons'),
    ((20,),         '20 neurons'),
    ((10, 10),      '10 + 10 (2 layers)'),
    ((20, 10, 5),   '20+10+5 (3 layers)'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, (layers, desc) in zip(axes.flat, architectures):
    nn = MLPClassifier(hidden_layer_sizes=layers, activation='relu',
                        max_iter=2000, random_state=42)
    nn.fit(X_moons, y_moons)
    n_params = sum(c.size for c in nn.coefs_) + sum(b.size for b in nn.intercepts_)
    plot_decision_boundary(nn, X_moons, y_moons, ax=ax,
                           title=f'{desc}\nAcc: {nn.score(X_moons, y_moons):.1%}, '
                                 f'Params: {n_params}')

plt.suptitle('How Architecture Shapes the Decision Boundary (Moons Dataset)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key observations:")
print("  • 3 sigmoid neurons = nearly linear (like logistic regression)")
print("  • 5 neurons = simple curves appear")
print("  • 10+ neurons = the network can trace complex shapes")
print("  • Multiple layers = even more complex (but diminishing returns here)")
print("\nNotice: 10 neurons already does great — more isn't always better!")

### How Does Width vs Depth Affect Learning?

**Width** (neurons per layer): More neurons in a single layer let the network represent more features *at the same level of abstraction*. Think of it as having more eyes looking at the same data from different angles.

**Depth** (number of layers): More layers let the network build *hierarchies of features*. Each layer can build on what the previous layer learned:
- Layer 1 might learn simple edges or thresholds
- Layer 2 combines those into shapes
- Layer 3 combines shapes into complex patterns

For most problems you'll encounter, **1-3 hidden layers** with **32-256 neurons each** is a reasonable starting point.

---

## Part 4: How Networks Learn — Forward Pass & Backpropagation

Training a neural network involves two phases that repeat thousands of times:

### Phase 1: Forward Pass
Data flows from input to output. At each layer:
1. Multiply inputs by weights and add bias: $z = W \vec{x} + \vec{b}$
2. Apply activation function: $\vec{a} = f(z)$
3. Pass the result to the next layer

At the end, compare the prediction to the true answer using a **loss function** (e.g., cross-entropy for classification, MSE for regression).

### Phase 2: Backward Pass (Backpropagation)
This is where learning happens. Working backward from the output:
1. Compute: "How much did each weight contribute to the error?"
2. This uses the **chain rule** from calculus to propagate gradients backward through every layer
3. Update each weight: $w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$

**This is exactly the gradient descent you already learned** — applied to every weight in the network simultaneously.

### Key Terms

| Term | Meaning |
|---|---|
| **Epoch** | One complete pass through the entire training dataset |
| **Batch** | A subset of training data used for one gradient update |
| **Learning rate** ($\eta$) | Step size for weight updates. Too large → overshoots. Too small → slow. |
| **Loss** | The error metric being minimized (lower = better) |
| **Convergence** | When the loss stops decreasing meaningfully |

In [ ]:
# ============================================================
# Walkthrough: forward pass with actual numbers
# ============================================================
print("═" * 65)
print("FORWARD PASS WALKTHROUGH — A tiny network with real numbers")
print("═" * 65)
print()
print("Network: 2 inputs → 2 hidden neurons (ReLU) → 1 output (sigmoid)")
print()

# Input
x = np.array([0.5, 0.8])
print(f"Step 0: Input features: x = {x}")
print()

# Hidden layer weights and biases (pretend these were learned)
W1 = np.array([[0.4, -0.3],
               [0.2,  0.6]])
b1 = np.array([0.1, -0.1])

# Step 1: Linear combination
z1 = W1 @ x + b1
print(f"Step 1: Hidden layer linear combination")
print(f"  z₁ = W₁·x + b₁ = {W1} · {x} + {b1}")
print(f"  z₁ = {z1}")
print()

# Step 2: Activation (ReLU)
a1 = np.maximum(0, z1)
print(f"Step 2: Apply ReLU activation")
print(f"  a₁ = ReLU({z1}) = {a1}")
print()

# Output layer
W2 = np.array([[0.7, -0.5]])
b2 = np.array([0.2])

z2 = W2 @ a1 + b2
print(f"Step 3: Output linear combination")
print(f"  z₂ = W₂·a₁ + b₂ = {W2} · {a1} + {b2}")
print(f"  z₂ = {z2}")
print()

# Sigmoid for output
output = 1 / (1 + np.exp(-z2))
print(f"Step 4: Apply sigmoid for final prediction")
print(f"  ŷ = sigmoid({z2[0]:.4f}) = {output[0]:.4f}")
print()

# Compare to true label
y_true = 1
loss = -(y_true * np.log(output[0]) + (1 - y_true) * np.log(1 - output[0]))
print(f"Step 5: Compute loss (cross-entropy)")
print(f"  True label: {y_true}")
print(f"  Prediction: {output[0]:.4f}")
print(f"  Loss: {loss:.4f}")
print()
print("Now backpropagation would compute gradients of this loss with respect")
print("to every weight and bias, then update them using gradient descent.")
print("This cycle repeats thousands of times until the loss converges.")

---

## Part 5: Watching a Network Train

Let's actually watch gradient descent happening inside a neural network — tracking the loss curve and seeing the decision boundary evolve over training.

In [ ]:
# ============================================================
# Animate training: loss curve + evolving boundary
# ============================================================
X_tr, X_val, y_tr, y_val = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42
)

# Manually step through training
nn_train = MLPClassifier(
    hidden_layer_sizes=(20, 10), activation='relu',
    solver='sgd', learning_rate_init=0.01, 
    max_iter=1, warm_start=True, random_state=42
)

train_losses = []
train_accs = []
val_accs = []
snapshots = {}  # save model state at certain epochs
snapshot_epochs = [1, 5, 20, 50, 150, 300]

for epoch in range(1, 301):
    nn_train.fit(X_tr, y_tr)
    train_losses.append(nn_train.loss_)
    train_accs.append(nn_train.score(X_tr, y_tr))
    val_accs.append(nn_train.score(X_val, y_val))
    
    if epoch in snapshot_epochs:
        # Save a copy of the model's predictions
        h = 0.05
        x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
        y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
        Z = nn_train.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        snapshots[epoch] = (xx, yy, Z)

# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Loss Curve — Gradient Descent at Work', fontweight='bold')
for ep in snapshot_epochs:
    axes[0].axvline(x=ep, color='gray', linestyle=':', alpha=0.5)

axes[1].plot(train_accs, label='Train', color='steelblue', linewidth=1.5)
axes[1].plot(val_accs, label='Validation', color='#e74c3c', linewidth=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Train vs Validation Accuracy', fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Show decision boundary snapshots over training
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_bold = ListedColormap(['#FF0000', '#0000FF'])

for ax, epoch in zip(axes.flat, snapshot_epochs):
    xx, yy, Z = snapshots[epoch]
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap=cmap_bold, 
              edgecolors='black', s=30, alpha=0.7)
    ax.set_title(f'Epoch {epoch} — Acc: {train_accs[epoch-1]:.1%}', 
                 fontweight='bold', fontsize=12)

plt.suptitle('Decision Boundary Evolution During Training', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Watch the boundary evolve:")
print("  Epoch 1 - 20:   Random — the network hasn't learned anything yet")
print("  Epoch 50:  Learned a little")
print("  Epoch 150: Nearly converged")
print("  Epoch 300: Final boundary — smooth and accurate")

---

## Part 6: Practical Considerations

Neural networks are powerful but fussy. Here are the practical issues you'll run into.

### 6.1 Feature Scaling is Non-Negotiable

Neural networks are **very** sensitive to the scale of input features. This isn't optional — it's a requirement. The reason: weight initialization and learning rates are calibrated for inputs near 0 with standard deviation near 1. If one feature ranges from 0-1 and another from 0-100,000, the network will struggle.

In [ ]:
# ============================================================
# Demonstrate: scaling is critical for neural networks
# ============================================================
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

# Without scaling
nn_unscaled = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
nn_unscaled.fit(X_tr_c, y_tr_c)
acc_unscaled = nn_unscaled.score(X_te_c, y_te_c)

# With scaling (using a pipeline — remember these from feature engineering?)
nn_scaled = Pipeline([
    ('scaler', StandardScaler()),
    ('nn', MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42))
])
nn_scaled.fit(X_tr_c, y_tr_c)
acc_scaled = nn_scaled.score(X_te_c, y_te_c)

print("Neural Network on Breast Cancer Data:")
print(f"  Without scaling: {acc_unscaled:.1%}")
print(f"  With scaling:    {acc_scaled:.1%}")
print(f"\n  Scaling improved accuracy by {(acc_scaled - acc_unscaled)*100:.1f} percentage points!")
print("\nWhy? Feature 'mean area' ranges ~100-2500 while 'mean smoothness'")
print("ranges ~0.05-0.16. Without scaling, the network essentially ignores")
print("small-scale features because their gradients are tiny.")

### 6.2 Overfitting: The Central Challenge

Neural networks have a lot of parameters. A network with just two hidden layers of 64 and 32 neurons, taking 30 features as input, has over 4,000 parameters. With only 450 training samples (breast cancer), that's a recipe for overfitting.

**Signs of overfitting:**
- Training accuracy is near 100% but test accuracy is significantly lower
- The gap between training and validation curves grows during training
- The loss on validation data starts *increasing* even as training loss decreases

**Remedies:**
- Use a simpler architecture (fewer layers, fewer neurons)
- Use **early stopping** — halt training when validation loss stops improving
- Use **regularization** (L2 penalty on weights, controlled by `alpha` in sklearn)
- Get more data (not always possible)

In [ ]:
# ============================================================
# Demonstrate overfitting: tiny vs. right-sized vs. huge network
# ============================================================
# Use the moons dataset — small enough to see overfitting clearly
X_small, y_small = make_moons(n_samples=100, noise=0.3, random_state=42)
X_s_tr, X_s_te, y_s_tr, y_s_te = train_test_split(
    X_small, y_small, test_size=0.3, random_state=42
)

configs = [
    ('Underfitting\n(too simple)', (2,), 0.0001),
    ('Good fit', (10, 5), 0.0001),
    ('Overfitting\n(too complex)', (200, 100, 50), 0.0001),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (title, layers, alpha) in zip(axes, configs):
    nn = MLPClassifier(hidden_layer_sizes=layers, activation='relu',
                        alpha=alpha, max_iter=2000, random_state=42)
    nn.fit(X_s_tr, y_s_tr)
    
    train_acc = nn.score(X_s_tr, y_s_tr)
    test_acc = nn.score(X_s_te, y_s_te)
    n_params = sum(c.size for c in nn.coefs_) + sum(b.size for b in nn.intercepts_)
    
    # Plot boundary using ALL data for visualization
    plot_decision_boundary(nn, X_small, y_small, ax=ax,
                           title=f'{title}\nTrain: {train_acc:.0%} | Test: {test_acc:.0%}\n'
                                 f'Params: {n_params:,}')

plt.suptitle('Underfitting → Good Fit → Overfitting', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Underfitting: Too simple to capture the pattern. Low train AND test accuracy.")
print("Good fit:     Captures the pattern without memorizing noise. Similar train & test.")
print("Overfitting:  Memorizes training data, including noise. High train, lower test.")
print("\nLook at the overfitting boundary — it contorts to classify every")
print("training point correctly, but those wiggles would hurt on new data.")

In [ ]:
# ============================================================
# Regularization: the alpha parameter
# ============================================================
# alpha adds penalty to weights, discouraging overly large values

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

alphas = [0.0001, 0.01, 0.1, 1.0]
for ax, alpha in zip(axes, alphas):
    nn = MLPClassifier(hidden_layer_sizes=(100, 50), activation='relu',
                        alpha=alpha, max_iter=2000, random_state=42)
    nn.fit(X_s_tr, y_s_tr)
    test_acc = nn.score(X_s_te, y_s_te)
    plot_decision_boundary(nn, X_small, y_small, ax=ax,
                           title=f'alpha = {alpha}\nTest acc: {test_acc:.0%}')

plt.suptitle('Effect of Regularization (alpha) — Smooths the Decision Boundary', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Small alpha → network is free to create complex boundaries (may overfit)")
print("Large alpha → weights are penalized, forcing simpler, smoother boundaries")
print("\nThis is the same bias-variance tradeoff you've seen in every model!")

### 6.3 Learning Rate

The learning rate ($\eta$) controls the step size in gradient descent. This is the same concept from when we studied gradient descent earlier — it just matters more now because we have thousands of weights being updated simultaneously.

In [ ]:
# ============================================================
# Learning rate: too small, just right, too large
# ============================================================
learning_rates = [0.0001, 0.01, 0.5]
lr_labels = ['Small (0.0001)', 'Medium (0.01)', 'Large (0.5)']
lr_colors = ['#3498db', '#2ecc71', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for lr, label, color in zip(learning_rates, lr_labels, lr_colors):
    nn = MLPClassifier(hidden_layer_sizes=(20, 10), activation='relu',
                        solver='sgd', learning_rate_init=lr,
                        max_iter=1, warm_start=True, random_state=42)
    losses = []
    accs = []
    for epoch in range(200):
        nn.fit(X_tr, y_tr)
        losses.append(nn.loss_)
        accs.append(nn.score(X_val, y_val))
    
    axes[0].plot(losses, label=label, color=color, linewidth=2)
    axes[1].plot(accs, label=label, color=color, linewidth=2)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 1.5)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Accuracy')
axes[1].set_title('Validation Accuracy', fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()



### 6.4 Quick Reference: Neural Network Hyperparameters

| Parameter | What It Controls | sklearn Name | Typical Values |
|---|---|---|---|
| Hidden layers | Network depth and width | `hidden_layer_sizes` | `(64, 32)`, `(128, 64, 32)` |
| Activation | Nonlinearity type | `activation` | `'relu'` (default, use this) |
| Learning rate | Gradient descent step size | `learning_rate_init` | 0.001 to 0.01 |
| Regularization | Overfitting control | `alpha` | 0.0001 to 0.1 |
| Max iterations | Training duration | `max_iter` | 200 to 1000 |
| Solver | Optimization algorithm | `solver` | `'adam'` (default, use this) |
| Early stopping | Stop when validation stops improving | `early_stopping` | `True` (recommended) |

---

## Part 7: Real Application — Handwritten Digit Recognition

Let's apply everything to a classic real-world task: recognizing handwritten digits (0-9).

The **sklearn digits dataset** contains 1,797 images of handwritten digits, each 8×8 pixels (64 features). This is a simplified version of the famous MNIST dataset.

This is a **multi-class classification** problem (10 classes), and it's where neural networks really start to show their strength.

In [ ]:
# ============================================================
# Load and explore the digits dataset
# ============================================================
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Dataset shape: {X_digits.shape}")
print(f"Each sample: {digits.images.shape[1]}×{digits.images.shape[2]} pixel image = {X_digits.shape[1]} features")
print(f"Feature values: pixel intensities from {X_digits.min():.0f} to {X_digits.max():.0f}")
print(f"Classes: digits 0-9")
print(f"Samples per class: ~{len(y_digits) // 10} each")

# Visualize some examples
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for digit in range(10):
    for row in range(2):
        idx = np.where(y_digits == digit)[0][row]
        axes[row, digit].imshow(digits.images[idx], cmap='gray_r')
        axes[row, digit].axis('off')
        if row == 0:
            axes[row, digit].set_title(str(digit), fontweight='bold', fontsize=14)

plt.suptitle('Handwritten Digit Samples (8×8 pixels)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nEach image is 8×8 = 64 pixels. The network receives these 64 pixel")
print("intensities as input features and must learn to classify them into 10 digits.")

In [ ]:
# ============================================================
# Train-test split
# ============================================================
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=42, stratify=y_digits
)

print(f"Training: {X_train_d.shape[0]} samples")
print(f"Test:     {X_test_d.shape[0]} samples")

In [ ]:
# ============================================================
# Compare architectures on digit recognition
# ============================================================
architectures = {
    'Small (32)':           (32,),
    'Medium (64, 32)':      (64, 32),
    'Large (128, 64, 32)':  (128, 64, 32),
    'Wide (256)':           (256,),
}

print("Architecture Comparison — Digit Recognition (5-fold CV):")
print("=" * 70)

arch_results = {}
for name, layers in architectures.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('nn', MLPClassifier(hidden_layer_sizes=layers, activation='relu',
                              max_iter=500, random_state=42))
    ])
    scores = cross_val_score(pipe, X_train_d, y_train_d, cv=5, scoring='accuracy')
    arch_results[name] = scores
    
    # Count parameters
    pipe.fit(X_train_d, y_train_d)
    nn = pipe.named_steps['nn']
    n_params = sum(c.size for c in nn.coefs_) + sum(b.size for b in nn.intercepts_)
    
    print(f"  {name:<25s}  Acc: {scores.mean():.4f} ± {scores.std():.4f}  "
          f"Params: {n_params:,}")

In [ ]:
# ============================================================
# Detailed evaluation of the best architecture
# ============================================================
best_nn = Pipeline([
    ('scaler', StandardScaler()),
    ('nn', MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu',
                          max_iter=500, random_state=42))
])
best_nn.fit(X_train_d, y_train_d)
y_pred_d = best_nn.predict(X_test_d)

print(f"Test Accuracy: {accuracy_score(y_test_d, y_pred_d):.4f}\n")
print("Classification Report:")
print(classification_report(y_test_d, y_pred_d))

# Confusion matrix
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test_d, y_pred_d, cmap='Blues', ax=ax
)
ax.set_title('Neural Network — Digit Recognition Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

print("The diagonal shows correct predictions. Off-diagonal entries show")
print("which digits get confused with each other (e.g., 3s and 8s look similar).")

In [ ]:
# ============================================================
# Visualize correct and incorrect predictions
# ============================================================
correct_mask = y_pred_d == y_test_d
incorrect_indices = np.where(~correct_mask)[0]
correct_indices = np.where(correct_mask)[0]

n_show = min(10, len(incorrect_indices))

if n_show > 0:
    fig, axes = plt.subplots(2, n_show, figsize=(16, 4))
    
    # Top row: correct predictions
    for i in range(n_show):
        idx = correct_indices[i]
        axes[0, i].imshow(X_test_d[idx].reshape(8, 8), cmap='gray_r')
        axes[0, i].set_title(f'pred: {y_pred_d[idx]}', fontsize=10, color='green')
        axes[0, i].axis('off')
    
    # Bottom row: incorrect predictions
    for i in range(n_show):
        idx = incorrect_indices[i]
        axes[1, i].imshow(X_test_d[idx].reshape(8, 8), cmap='gray_r')
        axes[1, i].set_title(f'pred: {y_pred_d[idx]} (true: {y_test_d[idx]})', 
                              fontsize=9, color='red')
        axes[1, i].axis('off')
    
    axes[0, 0].set_ylabel('Correct', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Wrong', fontsize=12, fontweight='bold')
    plt.suptitle('Neural Network Predictions on Test Digits', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    print(f"The network got {correct_mask.sum()} out of {len(y_test_d)} correct.")
    print("Look at the misclassified digits — many of them are genuinely hard to read!")
else:
    print("Perfect classification on the test set!")

---

## Part 8: Real Application — Breast Cancer Classification

The digits dataset is fun, but let's also see neural networks on a more "traditional" tabular dataset — the **Wisconsin Breast Cancer** dataset we used with SVMs. This has 30 features measured from cell nuclei, classifying tumors as malignant or benign.

In [ ]:
# ============================================================
# Neural network on breast cancer data
# ============================================================
print(f"Breast Cancer dataset: {X_cancer.shape[0]} samples, {X_cancer.shape[1]} features")
print(f"Classes: {dict(zip(cancer.target_names, np.bincount(y_cancer)))}\n")

# Try a few architectures
cancer_architectures = {
    'Small (16)':          (16,),
    'Medium (32, 16)':     (32, 16),
    'Large (64, 32, 16)':  (64, 32, 16),
}

print("Architecture Comparison — Breast Cancer (5-fold CV):")
print("=" * 65)

for name, layers in cancer_architectures.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('nn', MLPClassifier(hidden_layer_sizes=layers, activation='relu',
                              max_iter=500, random_state=42))
    ])
    scores = cross_val_score(pipe, X_tr_c, y_tr_c, cv=5, scoring='accuracy')
    print(f"  {name:<25s}  Acc: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# Final evaluation
best_cancer_nn = Pipeline([
    ('scaler', StandardScaler()),
    ('nn', MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu',
                          max_iter=500, random_state=42))
])
best_cancer_nn.fit(X_tr_c, y_tr_c)
y_pred_c = best_cancer_nn.predict(X_te_c)

print(f"Test Accuracy: {accuracy_score(y_te_c, y_pred_c):.4f}\n")
print(classification_report(y_te_c, y_pred_c, target_names=cancer.target_names))

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_te_c, y_pred_c, display_labels=cancer.target_names, cmap='Blues', ax=ax
)
ax.set_title('Neural Network — Breast Cancer Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

---

## Summary



| Concept | Key Idea |
|---|---|
| **Neuron** | Takes weighted inputs, adds bias, applies activation function. A single neuron = logistic regression. |
| **Activation function** | Introduces nonlinearity. Without it, stacking layers is pointless. ReLU is the default. |
| **Hidden layers** | Layers between input and output where patterns are learned. "Hidden" because you don't directly observe them. |
| **Architecture** | Number and size of hidden layers. More capacity → more complex patterns but more overfitting risk. |
| **Forward pass** | Data flows input → hidden → output. Compute prediction and loss. |
| **Backpropagation** | Chain rule + gradient descent applied to every weight. How the network actually learns. |
| **Learning rate** | Step size in gradient descent. Too small = slow, too large = unstable. |
| **Overfitting** | Network memorizes noise. Combat with simpler architectures, regularization (alpha), early stopping. |
| **Scaling** | Non-negotiable. Always use StandardScaler in a Pipeline for neural networks. |

### When to Use Neural Networks

| Good For | Not Ideal For |
|---|---|
| Complex nonlinear patterns | Very small datasets (<200 samples) |
| Image, text, sequence data | When interpretability is critical |
| Large datasets with many features | When training time is limited |
| When other models plateau | Simple, well-structured problems |

### Connection to pyMAISE

pyMAISE supports neural networks for nuclear engineering applications. The library handles much of the architecture search and training for you — but understanding neurons, layers, activation functions, and backpropagation is essential for diagnosing problems and making good modeling decisions.